# GSW–ICA component correlation analysis

This notebook examines the temporal association between EEG-derived generalised spike-and-wave (GSW) events and MELODIC ICA component time courses.

For each subject and run, GSW timings are converted from EEG samples to seconds, aligned to the fMRI repetition time, convolved with a canonical haemodynamic response function (HRF), and correlated with each ICA component using Pearson correlation. Components are then ranked by absolute correlation strength, and the full results and top five components per run are saved as CSV files.

**Important:** correlation indicates temporal association rather than causality. Positive and negative coefficients describe the direction of the association, while absolute correlation is used only for ranking.


# IMPORT NECESSARY PACKAGES

Import the libraries required for file parsing, numerical processing, statistical correlation, HRF convolution and path handling.


In [ ]:
# Regular expressions package are used to extract numerical onset and duration values from marker-file lines.
# NumPy and pandas support numerical processing and tabular output.
# Pearson correlation measures temporal association; gamma distributions are used to construct the HRF.
# FFT convolution applies the HRF to the binary GSW event vector.
# pathlib provides platform-independent file-path construction.

import re
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, gamma
from scipy.signal import fftconvolve
from pathlib import Path



# SETTINGS

Define the EEG sampling frequency, fMRI repetition time, input directories and the subject/run combinations included in the analysis.


In [ ]:
# EEG sampling frequency in hertz, used to convert marker samples to seconds.
eeg_fs = 250
# fMRI repetition time in seconds, used to align EEG events to fMRI volumes.
TR = 2.0

# Directory containing the EEG .Markers files.
base_eeg_dir = Path("/Users/phoebekusi-yeboah/Downloads/HCP_OUT/EEG")
# Directory containing the MELODIC mixing matrices.
base_melodic_dir = Path("/Users/phoebekusi-yeboah/Downloads/HCP_OUT/MELODIC")


# Explicitly map each subject/task/run combination to its corresponding MELODIC mixing-matrix filename.
runs = [
    {"sub": "sub-ga01", "task": "rest", "run": "run-01", "melodic": "sub-ga01_melodic_mix_rest1"},
    {"sub": "sub-ga01", "task": "rest", "run": "run-02", "melodic": "sub-ga01_melodic_mix_rest2"},
    {"sub": "sub-ga06", "task": "rest", "run": "run-01", "melodic": "sub-ga06_melodic_mix_rest1"},
    {"sub": "sub-ga06", "task": "rest", "run": "run-02", "melodic": "sub-ga06_melodic_mix_rest2"},
    {"sub": "sub-ga09", "task": "rest", "run": "run-01", "melodic": "sub-ga09_melodic_mix"},
    {"sub": "sub-ga12", "task": "rest", "run": "run-01", "melodic": "sub-ga12_melodic_mix_rest1"},
    {"sub": "sub-ga12", "task": "rest", "run": "run-02", "melodic": "sub-ga12_melodic_mix_rest2"},
    {"sub": "sub-ga12", "task": "cartoon", "run": "run-01", "melodic": "sub-ga12_melodic_mix_cartoon1"},
]

# FUNCTIONS

Define helper functions for generating the canonical HRF, reading GSW marker files and standardising time courses.


In [ ]:
# Generate a canonical HRF from a main response peak and delayed undershoot.
def spm_hrf(tr, duration=32.0):
    # Sample the HRF at the same temporal resolution as the fMRI data.
    t = np.arange(0, duration, tr)
    # Model the principal positive BOLD response.
    peak = gamma.pdf(t, 6)
    # Model the smaller delayed post-stimulus undershoot.
    undershoot = gamma.pdf(t, 16)
    # Combine the two gamma functions to form the canonical HRF shape.
    hrf = peak - undershoot / 6
    # Normalise the HRF before convolution.
    return hrf / np.sum(hrf)


# Read GSW onset and duration values from an EEG marker file and convert them from samples to seconds.
def read_gsw_markers(markers_path, eeg_fs=250):
    # Store each event as (onset_seconds, duration_seconds).
    events = []

    with open(markers_path, "r") as f:
        for line in f:
            line = line.strip()

            # Ignore annotations that are not labelled as GSW events.
            if "GSW" not in line.upper():
                continue

            # Extract numeric values from the current marker-file line.
            nums = re.findall(r"\d+\.?\d*", line)

            if len(nums) >= 2:
                start_sample = float(nums[0])
                duration_sample = float(nums[1])

                # Convert the marker onset from EEG samples to seconds.
                start_sec = start_sample / eeg_fs
                # Convert the marker duration from EEG samples to seconds.
                duration_sec = duration_sample / eeg_fs

                events.append((start_sec, duration_sec))

    return events


# Transform a time course to zero mean and unit variance before correlation.
def standardise(x):
    return (x - np.mean(x)) / np.std(x)

# MAIN LOOP 

Process each subject/run combination, construct the HRF-convolved GSW predictor and correlate it with every ICA component time course.


In [ ]:
# Accumulate one result row for every ICA component analysed.
all_results = []

# Process each predefined subject/task/run combination.
for item in runs:
    sub = item["sub"]
    task = item["task"]
    run = item["run"]

    print(f"\nProcessing {sub} {task} {run}")

    # Retrieve the mixing-matrix filename assigned to this run.
    melodic_filename = item["melodic"]

    # Construct the path to the EEG marker file for the current run.
    markers_file = base_eeg_dir / f"{sub}_ses-01_task-{task}_{run}.Markers"

    # Construct the path to the corresponding MELODIC mixing matrix.
    melodic_mix_file = base_melodic_dir / melodic_filename

    # Skip runs with a missing EEG marker file.
    if not markers_file.exists():
        print(f"Missing markers file: {markers_file}")
        continue

    # Skip runs with a missing ICA time-course file.
    if not melodic_mix_file.exists():
        print(f"Missing melodic_mix file: {melodic_mix_file}")
        continue

    # Load the MELODIC mixing matrix: rows are fMRI time points and columns are ICA component time courses.
    melodic_mix = np.loadtxt(melodic_mix_file)

    # Determine the number of fMRI volumes and ICA components.
    n_timepoints, n_components = melodic_mix.shape
    # Calculate the total fMRI run duration in seconds.
    run_duration = n_timepoints * TR

    # Load GSW onsets and durations from the EEG marker file.
    events = read_gsw_markers(markers_file, eeg_fs=eeg_fs)

    print(f"Loaded melodic_mix: {n_timepoints} timepoints x {n_components} ICs")
    print(f"Loaded {len(events)} GSW events")

    # A GSW predictor cannot be constructed when no events are present.
    if len(events) == 0:
        print("No GSW events found, skipping.")
        continue

    # Initialise a binary vector with one value per fMRI time point.
    # Time points overlapping a GSW event will be assigned a value of 1.
    gsw_vector = np.zeros(n_timepoints)

    # Map each EEG event interval onto the fMRI temporal grid.
    for start_sec, duration_sec in events:
        # Convert event onset from seconds to an fMRI volume index.
        start_idx = int(np.round(start_sec / TR))
        # Convert the event offset to an fMRI volume index.
        end_idx = int(np.round((start_sec + duration_sec) / TR))

        if start_idx < n_timepoints:
            # Prevent the event from extending beyond the final volume.
            end_idx = min(end_idx, n_timepoints - 1)
            # Mark all fMRI time points covered by the event.
            gsw_vector[start_idx:end_idx + 1] = 1

    # Generate the canonical HRF at the fMRI sampling interval.
    hrf = spm_hrf(TR)
    # Convolve GSW timing with the HRF to model the expected delayed BOLD response, then truncate it to the original run length.
    gsw_hrf = fftconvolve(gsw_vector, hrf)[:n_timepoints]
    # Standardise the predicted GSW-related BOLD time course.
    gsw_hrf = standardise(gsw_hrf)

    # Compare the GSW predictor with every ICA component time course.
    for i in range(n_components):
        # Standardise the current ICA component time course.
        ic_tc = standardise(melodic_mix[:, i])
        # Calculate Pearson's correlation coefficient and its p-value.
        r, p = pearsonr(gsw_hrf, ic_tc)

        # Store the component-level result. IC numbers are converted from zero-based Python indexing to one-based MELODIC numbering.
        all_results.append({
            "subject": sub,
            "task": task,
            "run": run,
            "IC": i + 1,
            "n_gsw_events": len(events),
            "correlation_r": r,
            "abs_correlation": abs(r),
            "p_value": p,
        })



Processing sub-ga01 rest run-01
Loaded melodic_mix: 300 timepoints x 72 ICs
Loaded 10 GSW events

Processing sub-ga01 rest run-02
Loaded melodic_mix: 300 timepoints x 81 ICs
Loaded 7 GSW events

Processing sub-ga06 rest run-01
Loaded melodic_mix: 300 timepoints x 56 ICs
Loaded 3 GSW events

Processing sub-ga06 rest run-02
Loaded melodic_mix: 300 timepoints x 72 ICs
Loaded 1 GSW events

Processing sub-ga09 rest run-01
Missing melodic_mix file: /Users/phoebekusi-yeboah/Downloads/HCP_OUT/MELODIC/sub-ga09_melodic_mix

Processing sub-ga12 rest run-01
Loaded melodic_mix: 300 timepoints x 61 ICs
Loaded 4 GSW events

Processing sub-ga12 rest run-02
Loaded melodic_mix: 300 timepoints x 78 ICs
Loaded 4 GSW events

Processing sub-ga12 cartoon run-01
Loaded melodic_mix: 300 timepoints x 86 ICs
Loaded 3 GSW events


# SAVE RESULTS

Combine, rank and export the component-level correlation results.


In [ ]:
# Convert all component-level results into a single table.
group_df = pd.DataFrame(all_results)

# Rank components within each run by absolute correlation strength, regardless of whether the association is positive or negative.
group_df = group_df.sort_values(
    ["subject", "task", "run", "abs_correlation"],
    ascending=[True, True, True, False]
)

# Save the complete set of component-level correlation results.
group_df.to_csv("/Users/phoebekusi-yeboah/Downloads/HCP_OUT/all_subjects_gsw_ic_correlations_3.csv", index=False)

# Retain the five ICA components with the largest absolute correlations for each subject/task/run combination.
top5_df = (
    group_df
    .sort_values(["subject", "task", "run", "abs_correlation"], ascending=[True, True, True, False])
    .groupby(["subject", "task", "run"])
    .head(5)
)

# Save the top-five component table used for network comparison.
top5_df.to_csv("/Users/phoebekusi-yeboah/Downloads/HCP_OUT/top5_ics_per_run_3.csv", index=False)

# Confirm the output files and display the top-ranked components.
print("\nSaved:")
print("all_subjects_gsw_ic_correlations.csv")
print("top5_ics_per_run.csv")

print("\nTop 5 ICs per run:")
print(top5_df)


Saved:
all_subjects_gsw_ic_correlations.csv
top5_ics_per_run.csv

Top 5 ICs per run:
      subject     task     run  IC  n_gsw_events  correlation_r  \
51   sub-ga01     rest  run-01  52            10       0.151968   
62   sub-ga01     rest  run-01  63            10      -0.145599   
48   sub-ga01     rest  run-01  49            10      -0.145435   
45   sub-ga01     rest  run-01  46            10       0.143724   
27   sub-ga01     rest  run-01  28            10      -0.129949   
79   sub-ga01     rest  run-02   8             7      -0.261127   
76   sub-ga01     rest  run-02   5             7      -0.241896   
105  sub-ga01     rest  run-02  34             7      -0.236664   
95   sub-ga01     rest  run-02  24             7      -0.203154   
72   sub-ga01     rest  run-02   1             7       0.198483   
205  sub-ga06     rest  run-01  53             3       0.230933   
191  sub-ga06     rest  run-01  39             3       0.163232   
207  sub-ga06     rest  run-01  55         